# pycorpdiff tutorial

**Comparative corpus analysis for modern Python workflows.**

This notebook is the canonical guided tour. It runs in CI on every push, so if a cell here has stale output, the package is broken — not the notebook.

We'll work with a small synthetic news corpus designed to expose two kinds of contrast:

1. **A frame contrast**: two outlets covering the same topic ('migrant') with engineered semantic fields — one humanising, one criminalising.
2. **A temporal contrast**: the same topic across nine years, with the discourse shifting around a 2018 policy event.

Both contrasts are deliberately stark — the point is to see the API at work on data where the signal is unambiguous.

## 1. Imports and version check

In [1]:
import pandas as pd
import pycorpdiff as pcd

print(f'pycorpdiff version: {pcd.__version__}')

pycorpdiff version: 0.1.0a0


## 2. Build the corpus

Two outlets (`humanising` and `criminalising`), nine years of coverage, eight documents per outlet per year.

In [2]:
humanising_templates = [
    'the migrant worker arrived and the migrant family settled here',
    'the migrant community grew and migrant workers thrived together',
    'the migrant family settled and the migrant community welcomed them',
    'the migrant worker and migrant rights advanced in the workplace',
    'the migrant family arrived seeking refuge and dignity',
    'the migrant community organised and the migrant worker spoke out',
    'the migrant family settled and migrant children attended school',
    'the migrant worker contributed and the migrant family flourished',
]
criminalising_templates = [
    'the migrant criminal threat and the migrant invasion grew worse',
    'the migrant threat and the migrant crime increased again',
    'the migrant invasion of migrant criminal gangs spread further',
    'the migrant criminal gangs and migrant invasion stayed dangerous',
    'the migrant threat persisted and the migrant criminal risk grew',
    'the migrant invasion and migrant criminal gangs threatened the border',
    'the migrant criminal element and the migrant threat alarmed residents',
    'the migrant gangs and migrant invasion narrative dominated coverage',
]

rows = []
for year in range(2015, 2024):
    for i, doc in enumerate(humanising_templates):
        rows.append({'text': doc, 'outlet': 'humanising', 'date': f'{year}-{(i%12)+1:02d}-15'})
    for i, doc in enumerate(criminalising_templates):
        rows.append({'text': doc, 'outlet': 'criminalising', 'date': f'{year}-{(i%12)+1:02d}-15'})

corpus = pcd.from_dataframe(
    pd.DataFrame(rows),
    text_col='text',
    meta_cols=('outlet', 'date'),
)
print(f'{len(corpus):,} documents · {corpus.total_tokens():,} tokens')
corpus.docs.head(3)

144 documents · 1,359 tokens


,text,outlet,date
0,the migrant worker arrived and the migrant fam...,humanising,2015-01-15
1,the migrant community grew and migrant workers...,humanising,2015-02-15
2,the migrant family settled and the migrant com...,humanising,2015-03-15


## 3. Slice the corpus

Slices remember their filters, so labels propagate into plots and result tables automatically.

In [3]:
humanising = corpus.slice(outlet='humanising')
criminalising = corpus.slice(outlet='criminalising')

print(f'{humanising.label}: {len(humanising)} docs, {humanising.total_tokens()} tokens')
print(f'{criminalising.label}: {len(criminalising)} docs, {criminalising.total_tokens()} tokens')

outlet='humanising': 72 docs, 675 tokens
outlet='criminalising': 72 docs, 684 tokens


## 4. Keyness: which words separate the two frames?

`compare(a, b).keyness()` returns one row per shared-vocabulary term with the signed log-likelihood (Dunning G²), Hardie LogRatio effect size, Gabrielatos %DIFF, Wilson Bayes factor, and Benjamini–Hochberg–adjusted *p*-values. Positive `g2` = overused in A.

In [4]:
keyness = pcd.compare(humanising, criminalising).keyness(min_count=3, dispersion=True)
print(keyness.summary())
keyness.table.head(12)

KeynessResult(log_likelihood, |a|=675, |b|=684, terms=53)


,term,count_a,count_b,expected_a,expected_b,g2,p_value,log_ratio,percent_diff,bayes_factor,dispersion_a,dispersion_b,dispersion_flag,p_adjusted
0,criminal,0,54,26.821192,27.178808,-74.147022,7.251040e-18,-6.749076,-100.0,3.421460e+14,0.000000,0.931132,True,3.843051e-16
1,family,45,0,22.350993,22.649007,62.981255,2.086832e-15,6.526903,inf,1.287052e+12,0.907205,0.000000,True,5.530104e-14
2,invasion,0,45,22.350993,22.649007,-61.789185,3.822682e-15,-6.488686,-100.0,7.091551e+11,0.000000,0.907754,True,6.753405e-14
3,worker,36,0,17.880795,18.119205,50.385004,1.263561e-12,6.208933,inf,2.367854e+09,0.881062,0.000000,True,1.674218e-11
4,threat,0,36,17.880795,18.119205,-49.431348,2.054365e-12,-6.170716,-100.0,1.469843e+09,0.000000,0.881062,True,1.814689e-11
5,gangs,0,36,17.880795,18.119205,-49.431348,2.054365e-12,-6.170716,-100.0,1.469843e+09,0.000000,0.881088,True,1.814689e-11
6,community,27,0,13.410596,13.589404,37.788753,7.883497e-10,5.800469,inf,4.356259e+06,0.846475,0.000000,True,5.222817e-09
7,settled,27,0,13.410596,13.589404,37.788753,7.883497e-10,5.800469,inf,4.356259e+06,0.846475,0.000000,True,5.222817e-09
8,arrived,18,0,8.940397,9.059603,25.192502,5.188352e-07,5.228562,inf,8.014429e+03,0.792758,0.000000,True,3.055363e-06
9,out,9,0,4.470199,4.529801,12.596251,3.865213e-04,4.267036,inf,1.474455e+01,0.686007,0.000000,True,4.749873e-04


### 4a. Volcano plot

x-axis is the LogRatio effect size, y-axis is significance. Top terms get labelled.

In [5]:
keyness.plot()

alt.LayerChart(...)

### 4b. Top-N bar chart

Sometimes a clean horizontal bar is more readable than the volcano.

In [6]:
keyness.plot(kind='bar', n=12)

alt.Chart(...)

### 4c. Explain a top result

Every Result carries references to its source corpora. `.explain(term)` pulls KWIC contexts from both sides.

In [7]:
keyness.explain('worker', n=3, window=3).table

,corpus,doc_id,position,left,keyword,right
0,outlet='humanising',0,2,the migrant,worker,arrived and the
1,outlet='humanising',3,2,the migrant,worker,and migrant rights
2,outlet='humanising',5,7,and the migrant,worker,spoke out


## 5. Collocation shift

What does each outlet put *next to* the word 'migrant'? `collocation_shift` measures the window-based co-occurrence in each corpus and reports `score_a - score_b` for every collocate.

In [8]:
shift = pcd.compare(humanising, criminalising).collocation_shift(
    'migrant', window=3, min_count=3, measure='logDice'
)
print(shift.summary())
shift.table.head(12)

CollocationShiftResult(target='migrant', measure=logDice, window=3, collocates=47)


,collocate,count_a,count_b,score_a,score_b,shift
0,invasion,0,63,6.912537,13.418829,-6.506292
1,family,54,0,13.268338,6.820091,6.448248
2,criminal,0,63,6.912537,13.352060,-6.439523
3,gangs,0,54,6.912537,13.268338,-6.355801
4,threat,0,54,6.912537,13.268338,-6.355801
5,settled,45,0,13.159066,6.820091,6.338976
6,worker,45,0,13.081530,6.820091,6.261439
7,community,36,0,12.841096,6.820091,6.021005
8,arrived,27,0,12.514573,6.820091,5.694482
9,contributed,18,0,12.029544,6.820091,5.209453


In [9]:
shift.plot(n=15)

alt.Chart(...)

### 5a. Explain a collocate

For collocation shifts, `.explain(collocate)` restricts the KWIC lines to windows in which both the target and the collocate appear — direct evidence for what's driving the shift.

In [10]:
shift.explain('criminal', n=3).table

,corpus,doc_id,position,left,keyword,right
0,outlet='criminalising',0,1,the,migrant,criminal threat and
1,outlet='criminalising',2,4,migrant invasion of,migrant,criminal gangs spread
2,outlet='criminalising',3,1,the,migrant,criminal gangs and


## 6. Temporal trajectories

`track(corpus, term).over_time()` returns a tidy frame with per-period relative frequencies and Wilson score confidence intervals. Multi-term tracking is supported in one call.

In [11]:
trajectory = pcd.track(corpus, ['worker', 'criminal', 'family']).over_time(
    freq='Y', time_col='date'
)
print(trajectory.summary())
trajectory.table.head(10)

TemporalTrajectory(targets=['worker', 'criminal', 'family'], freq='Y', periods=9)


,period,term,count,total,relfreq,ci_lower,ci_upper
0,2015,criminal,6,151,0.039735,0.018336,0.083972
1,2016,criminal,6,151,0.039735,0.018336,0.083972
2,2017,criminal,6,151,0.039735,0.018336,0.083972
3,2018,criminal,6,151,0.039735,0.018336,0.083972
4,2019,criminal,6,151,0.039735,0.018336,0.083972
5,2020,criminal,6,151,0.039735,0.018336,0.083972
6,2021,criminal,6,151,0.039735,0.018336,0.083972
7,2022,criminal,6,151,0.039735,0.018336,0.083972
8,2023,criminal,6,151,0.039735,0.018336,0.083972
9,2015,family,5,151,0.033113,0.014225,0.075166


In [12]:
trajectory.plot()

alt.LayerChart(...)

## 7. Before/after analysis

`compare.before_after(corpus, event_date=...)` splits the corpus chronologically and returns a regular `Comparison` you can keyness or collocation-shift just like the outlet comparison.

In [13]:
ba = pcd.compare.before_after(corpus, event_date='2019-01-01', time_col='date').keyness(
    min_count=5
)
print(ba.summary())
ba.table.head(8)

KeynessResult(log_likelihood, |a|=604, |b|=755, terms=53)


,term,count_a,count_b,expected_a,expected_b,g2,p_value,log_ratio,percent_diff,bayes_factor,p_adjusted
0,advanced,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
1,narrative,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
2,organised,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
3,out,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
4,persisted,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
5,refuge,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
6,residents,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0
7,rights,4,5,4.0,5.0,0.0,1.0,0.032421,0.0,0.027126,1.0


## 8. Where to go from here

What's *not* yet in this notebook:

- **Semantic shift** via Procrustes-aligned embeddings (Phase 6) — `compare(a, b).semantic_shift(target)`. Requires the `[semantic]` extra.
- **Changepoint detection** on temporal trajectories (Phase 7) — `trajectory.changepoints()`. Requires the `[temporal]` extra.
- **Multilingual corpora** via spaCy / Stanza / jieba — plug a different tokenizer into `Corpus(tokenizer=...)`.

Read the [README](../README.md) for the design philosophy, the [roadmap](../docs/roadmap.md) for what's next, and the [CHANGELOG](../CHANGELOG.md) for what just shipped.